# 24 — Quali connessioni normali generano falsi allarmi (T40)

Usiamo soltanto le predizioni salvate sul test ufficiale. Un **falso allarme** è
una connessione Normal classificata come una delle quattro famiglie di attacco.
Analisi descrittiva: nessun nuovo training e nessuna scelta di parametri sul test.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support, f1_score, accuracy_score

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data import COLUMN_NAMES
from src.targets import AUTHOR_ATTACK_TO_FAMILY, FAMILY_ORDER
OUTPUT = ROOT / 'artifacts/false_alarm_diagnosis'
OUTPUT.mkdir(parents=True, exist_ok=True)
protocol_path = ROOT / 'configs/official_test_v1.json'
protocol = json.loads(protocol_path.read_text(encoding='utf-8'))
protocol_hash = hashlib.sha256(protocol_path.read_bytes()).hexdigest()
for name, digest in protocol['input_sha256'].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == digest
test = pd.read_csv(ROOT / 'data/raw/KDDTest+.csv', header=None, names=COLUMN_NAMES)
assert test['attack_label'].isin(AUTHOR_ATTACK_TO_FAMILY).all()
expected_truth = test['attack_label'].map(AUTHOR_ATTACK_TO_FAMILY).to_numpy(dtype=str)
models = {
    'decision_tree': ('Decision Tree', 'decision_tree_official_test'),
    'random_forest_balanced': ('Random Forest', 'random_forest_balanced_official_test'),
    'gaussian_naive_bayes': ('Gaussian Naive Bayes', 'gaussian_naive_bayes_official_test'),
    'svm_rbf': ('SVM RBF', 'svm_rbf_official_test'),
    'neural_network': ('Rete neurale', 'neural_network_official_test'),
    'kmeans': ('K-Means', 'kmeans_official_test'),
}
train = pd.read_csv(ROOT / 'data/raw/KDDTrain+.csv', header=None, names=COLUMN_NAMES)
assert train['attack_label'].isin(AUTHOR_ATTACK_TO_FAMILY).all()

## 1. Verificare e caricare le predizioni

Controlliamo protocollo, indici, target e metriche prima di unire le predizioni alle feature originali.

In [ ]:
prediction_frames = []
overall_rows = []
class_rows = []
matrices = {}
saved_results = {}
for model_id, (label, artifact) in models.items():
    folder = ROOT / 'artifacts' / artifact
    result = json.loads((folder / 'results.json').read_text(encoding='utf-8'))
    assert result['protocol_sha256'] == protocol_hash
    assert result['protocol_id'] == protocol['protocol_id']
    assert result['input_sha256'] == protocol['input_sha256']
    assert result['label_profile'] == protocol['label_profile']
    assert result['family_order'] == list(FAMILY_ORDER)
    assert result['train_rows'] == 125973 and result['test_rows'] == len(expected_truth)
    assert result['classifier_parameters'] == protocol['models'][model_id]['parameters']
    assert hashlib.sha256((folder / 'predictions.npz').read_bytes()).hexdigest() == result['predictions_sha256']
    with np.load(folder / 'predictions.npz', allow_pickle=False) as saved:
        truth, predicted = saved['truth'], saved['predicted']
        np.testing.assert_array_equal(saved['test_indices'], np.arange(len(expected_truth)))
    np.testing.assert_array_equal(truth, expected_truth)
    assert set(predicted) <= set(FAMILY_ORDER)
    counts = confusion_matrix(truth, predicted, labels=FAMILY_ORDER)
    normalized = confusion_matrix(truth, predicted, labels=FAMILY_ORDER, normalize='true')
    np.testing.assert_array_equal(counts, result['confusion_counts'])
    np.testing.assert_allclose(normalized, result['confusion_normalized'])
    precision, recall, f1, support = precision_recall_fscore_support(truth, predicted, labels=FAMILY_ORDER, zero_division=0)
    scores = {
        'macro_f1': f1_score(truth, predicted, labels=FAMILY_ORDER, average='macro', zero_division=0),
        'weighted_f1': f1_score(truth, predicted, labels=FAMILY_ORDER, average='weighted', zero_division=0),
        'accuracy': accuracy_score(truth, predicted),
    }
    for metric, value in scores.items():
        np.testing.assert_allclose(value, result['metrics'][metric])
    overall_rows.append({'model_id': model_id, 'model': label, **scores, 'fit_seconds': result['fit_seconds']})
    for i, family in enumerate(FAMILY_ORDER):
        class_rows.append({'model_id': model_id, 'model': label, 'family': family,
                           'precision': precision[i], 'recall': recall[i], 'f1': f1[i],
                           'support': int(support[i]), 'true_positives': int(counts[i,i]),
                           'false_positives': int(counts[:,i].sum()-counts[i,i]),
                           'error_rate': 1-recall[i]})
    prediction_frames.append(pd.DataFrame({
        'model_id': model_id, 'model': label, 'test_index': np.arange(len(truth)),
        'attack': test['attack_label'].to_numpy(), 'family': truth, 'predicted': predicted,
    }))
    matrices[model_id] = normalized
    saved_results[model_id] = result
overall = pd.DataFrame(overall_rows)
per_class = pd.DataFrame(class_rows)
predictions = pd.concat(prediction_frames, ignore_index=True)

## 2. Quanti falsi allarmi e verso quali famiglie

Il denominatore è sempre il numero di connessioni realmente normali (9.711).
Questi conteggi non includono gli attacchi assegnati a una famiglia sbagliata.

In [ ]:
normal = predictions.loc[predictions['family'].eq('Normal')].copy()
normal['false_alarm'] = normal['predicted'].ne('Normal')
summary = normal.groupby(['model_id', 'model'], sort=False).agg(
    normal_support=('false_alarm', 'size'), false_alarms=('false_alarm', 'sum')).reset_index()
summary['false_alarm_rate'] = summary['false_alarms'] / summary['normal_support']
destinations = pd.crosstab(normal['model'], normal['predicted']).reindex(
    index=[label for label, _ in models.values()], columns=FAMILY_ORDER, fill_value=0)
assert summary['normal_support'].eq(9711).all()
assert destinations.sum(axis=1).eq(9711).all()
display(summary.round(4))
display(destinations)

## 3. Protocollo, servizio e flag

Per ogni gruppo mostriamo supporto, falsi allarmi e tasso: un servizio frequente
può avere molti errori senza avere il tasso più alto. I gruppi piccoli restano
nelle tabelle complete; il grafico usa almeno 100 connessioni normali per gruppo.
`flag` descrive lo stato della connessione, non l'etichetta di attacco.
Le associazioni osservate non identificano la causa delle decisioni del modello.

In [ ]:
categorical = ['protocol_type', 'service', 'flag']
normal = normal.join(test[categorical], on='test_index', validate='many_to_one')
group_frames = []
for feature in categorical:
    grouped = normal.groupby(['model_id', 'model', feature], sort=False).agg(
        normal_support=('false_alarm', 'size'), false_alarms=('false_alarm', 'sum')).reset_index()
    grouped = grouped.rename(columns={feature: 'value'})
    grouped['feature'] = feature
    grouped['false_alarm_rate'] = grouped['false_alarms'] / grouped['normal_support']
    group_frames.append(grouped)
groups = pd.concat(group_frames, ignore_index=True)
for (_, _), group in groups.groupby(['model_id', 'feature']):
    assert group['normal_support'].sum() == 9711
nb_groups = groups.loc[groups['model_id'].eq('gaussian_naive_bayes')]
display(nb_groups.sort_values(['feature', 'false_alarms'], ascending=[True, False]).round(4))

In [ ]:
assets = ROOT / 'docs/assets/false_alarm_diagnosis'
assets.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout='constrained')
labels = ['DT', 'RF', 'GaussianNB', 'SVM', 'MLP', 'K-Means']
bars = axes[0].bar(labels, summary['false_alarm_rate'] * 100, color='#347fb5')
axes[0].bar_label(bars, fmt='%.1f%%', padding=3)
axes[0].set(title='Connessioni normali segnalate come attacco', ylabel='% delle 9.711 connessioni normali', ylim=(0, 100))
services = nb_groups.loc[nb_groups['feature'].eq('service') & nb_groups['normal_support'].ge(100)].nlargest(8, 'false_alarms')
bars = axes[1].barh(services['value'], services['false_alarm_rate'] * 100, color='#ed912a')
axes[1].bar_label(bars, labels=[f'{int(r.false_alarms)}/{int(r.normal_support)}' for r in services.itertuples()], padding=3)
axes[1].invert_yaxis()
axes[1].set(title='Naive Bayes: falsi allarmi per servizio\nSolo gruppi con almeno 100 connessioni normali', xlabel='% dei normali del servizio (conteggio accanto)', xlim=(0, 125))
fig.savefig(OUTPUT / 'false_alarm_rates.png', dpi=160, bbox_inches='tight')
fig.savefig(assets / 'false_alarm_rates.png', dpi=160, bbox_inches='tight')
plt.show()

## 4. Gli stessi normali vengono segnalati da più modelli?

Contiamo quanti dei sei modelli segnalano ogni connessione normale.
L'accordo tra modelli non cambia l'etichetta vera del dataset.

In [ ]:
agreement = normal.pivot(index='test_index', columns='model_id', values='false_alarm')
assert agreement.shape == (9711, 6) and not agreement.isna().any().any()
votes = agreement.sum(axis=1)
overlap = votes.value_counts().reindex(range(7), fill_value=0).rename_axis('models_flagging').reset_index(name='normal_connections')
nb_only = agreement['gaussian_naive_bayes'] & votes.eq(1)
display(overlap)
print('Falsi allarmi esclusivi di Naive Bayes:', int(nb_only.sum()))

## 5. Distribuzioni numeriche: confronto descrittivo

Per Naive Bayes confrontiamo mediane e quartili dei normali corretti e dei falsi
allarmi su tutte le feature numeriche. Non sono importanze delle feature né una
dimostrazione che rimuovere una colonna migliori il modello. Non usiamo questo
test per selezionare feature o regolare parametri.

In [ ]:
nb_normal = normal.loc[normal['model_id'].eq('gaussian_naive_bayes')]
numeric_features = [c for c in test.columns if c not in categorical + ['attack_label', 'difficulty'] and pd.api.types.is_numeric_dtype(test[c])]
# Solo le 38 feature numeriche originali: mai etichetta o difficulty level.
numeric_features = [c for c in numeric_features if 'difficulty' not in c]
assert len(numeric_features) == 38
numeric = test.loc[nb_normal['test_index'], numeric_features].copy()
numeric['group'] = np.where(nb_normal['false_alarm'].to_numpy(), 'Falso allarme', 'Normal corretto')
profiles = numeric.groupby('group')[numeric_features].quantile([0.25, 0.5, 0.75]).rename_axis(['group', 'quantile'])
display(profiles.T)
summary.to_csv(OUTPUT / 'model_false_alarms.csv', index=False)
destinations.to_csv(OUTPUT / 'normal_prediction_destinations.csv')
groups.to_csv(OUTPUT / 'categorical_groups.csv', index=False)
overlap.to_csv(OUTPUT / 'model_agreement.csv', index=False)
profiles.to_csv(OUTPUT / 'nb_numeric_profiles.csv')
(OUTPUT / 'verification.json').write_text(json.dumps({
    'protocol_sha256': protocol_hash, 'normal_rows': 9711, 'models': 6,
    'nb_only_false_alarms': int(nb_only.sum()), 'new_fits': 0,
}, indent=2), encoding='utf-8')